In [ ]:
!git clone https://github.com/Ish2905/Comemo-Dataset.git

In [ ]:
%cd /content/Comemo-Dataset

In [ ]:
!git pull

In [ ]:
# ==== DO NOT MODIFY THIS CELL ====
from google.colab import drive
drive.mount('/content/drive')

import duckdb
import os

DB_PATH = "/content/drive/MyDrive/Capstone/comemo.db"

# HARD FAIL if Drive is not mounted
assert os.path.exists("/content/drive/MyDrive"), "Drive not mounted!"

# HARD FAIL if DB file missing (after first creation)
if not os.path.exists(DB_PATH):
    print("⚠️ comemo.db not found yet (first run only)")
else:
    print("✅ Using existing database:", DB_PATH)

con = duckdb.connect(DB_PATH)

# Sanity check
print(con.execute("SHOW TABLES").fetchdf())
# =================================


In [ ]:
con.execute("""
SELECT
  'reviews_raw' AS table,
  COUNT(*) AS rows
FROM reviews_raw
UNION ALL
SELECT
  'metadata_raw',
  COUNT(*)
FROM metadata_raw
""").fetchdf()


In [ ]:
con.execute("SHOW TABLES").fetchdf()



In [ ]:
con.execute("""
CREATE TABLE metadata_raw AS
SELECT
    parent_asin,
    main_category,
    title,
    features,
    description,
    details,
    categories,
    store,
    price,
    TRY_CAST(average_rating AS DOUBLE) AS average_rating,
    rating_number
FROM read_json(
    '/content/drive/MyDrive/Capstone/comemo_data/metadata.jsonl',
    format = 'newline_delimited',
    sample_size = -1, -- Changed from 0 to -1 to read all input for schema detection
    columns = {
        parent_asin: 'VARCHAR',
        main_category: 'VARCHAR',
        title: 'VARCHAR',
        features: 'JSON',
        description: 'JSON',
        details: 'JSON',
        categories: 'JSON',
        store: 'VARCHAR',
        price: 'VARCHAR',
        average_rating: 'VARCHAR',
        rating_number: 'BIGINT'
    },
    ignore_errors = true
)
WHERE parent_asin IS NOT NULL;
""")

In [ ]:
con.execute("DESC metadata_raw").fetchdf()

In [ ]:
con.execute("""
CREATE TABLE reviews_raw AS
SELECT
    parent_asin,
    rating,
    title,
    text,
    timestamp,
    verified_purchase,
    helpful_vote
FROM read_json(
    '/content/drive/MyDrive/Capstone/comemo_data/reviews.jsonl',
    format = 'newline_delimited'
)
WHERE parent_asin IS NOT NULL;
""")


In [ ]:
con.execute("DESC reviews_raw").fetchdf()

In [ ]:
con.close()
print("DuckDB connection closed. Tables should be persisted to /content/drive/MyDrive/Capstone/comemo.db")

In [ ]:
con.execute("""
COPY reviews_raw
TO '/content/drive/MyDrive/Capstone/reviews_raw.parquet'
(FORMAT PARQUET);

COPY metadata_raw
TO '/content/drive/MyDrive/Capstone/metadata_raw.parquet'
(FORMAT PARQUET);
""")

In [ ]:
con.execute("SHOW TABLES;").fetchdf()